# Final Inference & Forensic Analysis

**Experimental training and model selection were completed previously. This notebook performs
inference and forensic analysis using the frozen B + QuantileTransformer seed-42 model and its
saved preprocessing artifacts.**

No model is trained, no scaler is fit, and no imputer is fit in this notebook. All preprocessing
objects and model weights are loaded exactly as saved in
`outputs/gnn_B_quantile_seed42/`.

## 0. Imports and paths

In [1]:
import time, os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import joblib

t0 = time.time()
def log(msg): print(f"[{time.time()-t0:7.1f}s] {msg}", flush=True)

DATA_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++"
PROCESSED_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\processed"
ARTIFACT_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\gnn_B_quantile_seed42"
OUTPUTS_DIR = r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs"
print("Paths configured.")

Paths configured.


## Step 1 — Load final model

Loads the frozen GraphSAGE checkpoint (`model_checkpoint.pt`), recreates only the architecture
needed to hold the saved weights, loads the state_dict, and sets the model to `eval()` mode.
No training occurs here.

In [2]:
print("="*70)
print("STEP 1: LOAD FINAL MODEL")
print("="*70)

required_files = ["model_checkpoint.pt", "quantile_transformer.joblib", "median_imputation.joblib",
                   "feature_columns.json", "scaling_columns.json", "binary_columns.json", "metadata.json"]
for fn in required_files:
    fp = os.path.join(ARTIFACT_DIR, fn)
    if not os.path.exists(fp):
        raise FileNotFoundError(f"STOP: required artifact missing: {fp}")
print("All 7 required artifact files found.")

checkpoint = torch.load(os.path.join(ARTIFACT_DIR, "model_checkpoint.pt"), weights_only=False)

class GraphSAGENet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index); h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)

arch = checkpoint["architecture"]
model = GraphSAGENet(in_dim=arch["in_dim"], hidden_dim=arch["hidden_dim"], out_dim=arch["out_dim"], dropout=arch["dropout"])
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Checkpoint loaded: {os.path.join(ARTIFACT_DIR, 'model_checkpoint.pt')}")
print(f"  Architecture: {arch}")
print(f"  n_features (checkpoint): {checkpoint['n_features']}")
print(f"  training_seed: {checkpoint['training_seed']}")
print(f"  best_epoch: {checkpoint['best_epoch']}")
print(f"  selected_threshold: {checkpoint['selected_threshold']}")
print(f"  class_weights: {checkpoint['class_weights']}")
print(f"  val_pr_auc: {checkpoint['val_pr_auc']:.4f}  val_roc_auc: {checkpoint['val_roc_auc']:.4f}")
print(f"  test_pr_auc: {checkpoint['test_pr_auc']:.4f}  test_roc_auc: {checkpoint['test_roc_auc']:.4f}")
print(f"Model set to eval() mode: {not model.training}")

STEP 1: LOAD FINAL MODEL
All 7 required artifact files found.
Checkpoint loaded: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\gnn_B_quantile_seed42\model_checkpoint.pt
  Architecture: {'type': 'GraphSAGENet', 'conv1': 'SAGEConv', 'conv2': 'SAGEConv', 'in_dim': 186, 'hidden_dim': 64, 'out_dim': 2, 'dropout': 0.3}
  n_features (checkpoint): 186
  training_seed: 42
  best_epoch: 81
  selected_threshold: 0.9
  class_weights: [0.5654888153076172, 4.317446708679199]
  val_pr_auc: 0.8923  val_roc_auc: 0.9660
  test_pr_auc: 0.5063  test_roc_auc: 0.8371
Model set to eval() mode: True


## Step 2 — Load saved preprocessing

The `QuantileTransformer` and median-imputation values were already fit during training.
They are loaded with `joblib.load()` only — `.fit()` / `.fit_transform()` are never called here.

In [3]:
print("\n" + "="*70)
print("STEP 2: LOAD SAVED PREPROCESSING (already fit -- NOT re-fit)")
print("="*70)

qt = joblib.load(os.path.join(ARTIFACT_DIR, "quantile_transformer.joblib"))
median_artifact = joblib.load(os.path.join(ARTIFACT_DIR, "median_imputation.joblib"))
augmented_medians = pd.Series(median_artifact["augmented_medians"])
new_feature_medians = pd.Series(median_artifact["new_feature_medians"])
augmented_features = median_artifact["augmented_features"]
continuous_new = median_artifact["continuous_new_features"]

with open(os.path.join(ARTIFACT_DIR, "feature_columns.json")) as f:
    saved_feature_cols = json.load(f)
with open(os.path.join(ARTIFACT_DIR, "scaling_columns.json")) as f:
    saved_scaling_cols = json.load(f)
with open(os.path.join(ARTIFACT_DIR, "binary_columns.json")) as f:
    saved_binary_cols = json.load(f)
with open(os.path.join(ARTIFACT_DIR, "metadata.json")) as f:
    metadata = json.load(f)

VALIDATED_THRESHOLD = metadata["selected_threshold"]
assert hasattr(qt, "quantiles_"), "Loaded object is not a fitted QuantileTransformer!"
print(f"QuantileTransformer loaded (already fit -- .fit()/.fit_transform() NOT called).")
print(f"  n_quantiles_: {qt.n_quantiles_}  output_distribution: {qt.output_distribution}")
print(f"Median-imputation values loaded: {len(augmented_medians)} augmented + {len(new_feature_medians)} new-feature medians")
print(f"Saved feature_columns: {len(saved_feature_cols)}  scaling_columns: {len(saved_scaling_cols)}  binary_columns: {saved_binary_cols}")
print(f"Validated decision threshold (from metadata.json): {VALIDATED_THRESHOLD}")


STEP 2: LOAD SAVED PREPROCESSING (already fit -- NOT re-fit)
QuantileTransformer loaded (already fit -- .fit()/.fit_transform() NOT called).
  n_quantiles_: 1000  output_distribution: normal
Median-imputation values loaded: 17 augmented + 13 new-feature medians
Saved feature_columns: 186  scaling_columns: 183  binary_columns: ['augmented_features_missing', 'STR2', 'TMP6_is_first_appearance']
Validated decision threshold (from metadata.json): 0.9


## Step 4 — Load Elliptic++ data and graph

Reconstructed identically to the validated training notebook (`04_GNN_Baseline.ipynb`): same raw
files, same `txId`-sorted node ordering, same directed edge list, same class encoding. Loaded here
(ahead of Step 3's feature-matrix construction) because building the B feature matrix requires
`tx_data`. Class 3 (unknown) transactions are identified but never treated as ground truth.

In [4]:
print("\n" + "="*70)
print("STEP 4: LOAD ELLIPTIC++ DATA AND GRAPH")
print("="*70)

tx_features = pd.read_csv(DATA_DIR + r"\txs_features.csv")
tx_classes = pd.read_csv(DATA_DIR + r"\txs_classes.csv")
edges_raw = pd.read_csv(DATA_DIR + r"\txs_edgelist.csv")
assert tx_features.shape == (203769, 184)
assert tx_classes.shape[0] == 203769
assert edges_raw.shape == (234355, 2)

tx_data = tx_features.merge(tx_classes, on="txId", how="left")
assert len(tx_data) == 203769
assert tx_data["class"].isna().sum() == 0
tx_data = tx_data.sort_values("txId", kind="mergesort").reset_index(drop=True)

model_feature_cols = [c for c in tx_features.columns if c != "txId"]
assert len(model_feature_cols) == 183

txid_to_node = {txid: i for i, txid in enumerate(tx_data["txId"].values)}
n_nodes = len(tx_data)
assert n_nodes == 203769

edge_cols = edges_raw.columns.tolist()
src = edges_raw[edge_cols[0]].map(txid_to_node).values
dst = edges_raw[edge_cols[1]].map(txid_to_node).values
assert not np.isnan(src).any() and not np.isnan(dst).any()
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
assert edge_index.shape[1] == 234355
print(f"Transaction nodes: {n_nodes} (expected 203,769)")
print(f"Transaction edges: {edge_index.shape[1]} (expected 234,355)")

class_counts = tx_data["class"].value_counts().sort_index()
print(f"Class distribution: {class_counts.to_dict()}")
n_class3 = int((tx_data["class"] == 3).sum())
print(f"Class-3 (unknown) count: {n_class3} (expected ~157,205)")

time_step_arr = tx_data["Time step"].values
def split_of_step(t):
    if t <= 34: return 0
    elif t <= 41: return 1
    else: return 2
split_id = np.array([split_of_step(t) for t in time_step_arr])

known_mask_all = tx_data["class"].isin([1, 2]).values
y_all = np.where(tx_data["class"].values == 1, 1, np.where(tx_data["class"].values == 2, 0, -1))
train_mask_np = (split_id == 0) & known_mask_all
val_mask_np = (split_id == 1) & known_mask_all
test_mask_np = (split_id == 2) & known_mask_all
assert train_mask_np.sum() == 29894 and val_mask_np.sum() == 7829 and test_mask_np.sum() == 8841
y = torch.tensor(y_all, dtype=torch.long)
test_mask = torch.tensor(test_mask_np, dtype=torch.bool)
class3_mask_np = (tx_data["class"] == 3).values
assert class3_mask_np.sum() == n_class3
print("[PASS] Graph, labels, and masks reconstructed identically to the validated training notebook.")


STEP 4: LOAD ELLIPTIC++ DATA AND GRAPH


Transaction nodes: 203769 (expected 203,769)
Transaction edges: 234355 (expected 234,355)
Class distribution: {1: 4545, 2: 42019, 3: 157205}
Class-3 (unknown) count: 157205 (expected ~157,205)
[PASS] Graph, labels, and masks reconstructed identically to the validated training notebook.


## Step 3 — Build final B feature matrix (186 features)

Uses the exact B feature definition (A2 + 14 transaction-time-safe features + the TMP6
first-appearance indicator), in the exact column order stored in `feature_columns.json`. Missing
values are filled using the **saved** train-only medians, and the **saved** (already-fit)
`QuantileTransformer` is applied with `.transform()` only. If the constructed feature ordering or
column lists do not match the saved artifacts exactly, this cell raises and stops rather than
silently continuing.

In [5]:
print("\n" + "="*70)
print("STEP 3: BUILD FINAL B FEATURE MATRIX (186 features, saved artifacts only)")
print("="*70)

REMOVE_13 = [
    "Local_feature_49", "Local_feature_65", "Local_feature_30", "Local_feature_29",
    "Local_feature_23", "Local_feature_47", "Local_feature_25", "Local_feature_43",
    "Local_feature_41", "Local_feature_54", "Local_feature_53", "Local_feature_55",
    "Local_feature_18",
]
a2_cols = [c for c in model_feature_cols if c not in REMOVE_13]
assert len(a2_cols) == 170

new_feats = pd.read_parquet(os.path.join(PROCESSED_DIR, "transaction_time_safe_features_v1.parquet"))
NEW_14 = ["FIN3","FIN4","FIN5","FIN6","STR1","STR2","TMP1","TMP2","TMP4","TMP5","TMP6","TMP7","NET1","NET2"]

merged = tx_data[["txId"] + a2_cols].merge(
    new_feats[["txId"] + NEW_14 + ["TMP6_is_first_appearance"]], on="txId", how="left"
)
assert len(merged) == len(tx_data)
assert (merged["txId"].values == tx_data["txId"].values).all()
merged["augmented_features_missing"] = merged[augmented_features].isna().all(axis=1).astype(int)

final_feature_cols_b = a2_cols + ["augmented_features_missing"] + NEW_14 + ["TMP6_is_first_appearance"]
assert len(final_feature_cols_b) == 186

# CRITICAL CHECK: constructed feature ordering must exactly match the saved artifact
if final_feature_cols_b != saved_feature_cols:
    raise AssertionError("STOP: constructed feature ordering does NOT match feature_columns.json -- mismatch, not silently fixing.")
print("[PASS] Constructed 186-column feature ordering matches feature_columns.json exactly.")

scaling_cols_b = a2_cols + [c for c in NEW_14 if c != "STR2"]
if scaling_cols_b != saved_scaling_cols:
    raise AssertionError("STOP: constructed scaling-column list does NOT match scaling_columns.json.")
binary_cols_b = ["augmented_features_missing", "STR2", "TMP6_is_first_appearance"]
if binary_cols_b != saved_binary_cols:
    raise AssertionError("STOP: constructed binary-column list does NOT match binary_columns.json.")
print("[PASS] Scaling-column list and binary-column list match saved artifacts exactly.")

# Raw, pre-scaling copy retained for forensic indicator reporting later
merged_raw = merged.copy()

# Apply SAVED median-imputation values (NOT recomputed)
X_b = merged.copy()
for col in augmented_features:
    X_b[col] = X_b[col].fillna(augmented_medians[col])
for col in continuous_new:
    X_b[col] = X_b[col].fillna(new_feature_medians[col])
assert X_b[final_feature_cols_b].isna().sum().sum() == 0
print("[PASS] Missing values filled using SAVED train-only medians (not recomputed).")

# Apply SAVED (already-fit) QuantileTransformer -- .transform() ONLY
X_b_q = X_b.copy()
X_b_q[scaling_cols_b] = X_b_q[scaling_cols_b].astype("float64")
X_b_q[scaling_cols_b] = qt.transform(X_b[scaling_cols_b])
X_b_q_final = X_b_q[final_feature_cols_b].astype("float32")
x_infer = torch.tensor(X_b_q_final.values, dtype=torch.float32)
assert x_infer.shape == (203769, 186)
print(f"Final inference feature tensor shape: {x_infer.shape} (expected [203769, 186])")

assert not torch.isnan(x_infer).any().item(), "NaN in inference input!"
assert not torch.isinf(x_infer).any().item(), "Inf in inference input!"
print("[PASS] No NaN/Inf in inference model inputs.")


STEP 3: BUILD FINAL B FEATURE MATRIX (186 features, saved artifacts only)


[PASS] Constructed 186-column feature ordering matches feature_columns.json exactly.
[PASS] Scaling-column list and binary-column list match saved artifacts exactly.


[PASS] Missing values filled using SAVED train-only medians (not recomputed).


Final inference feature tensor shape: torch.Size([203769, 186]) (expected [203769, 186])
[PASS] No NaN/Inf in inference model inputs.


## Step 5 — Verify final pipeline

Before touching Class-3 transactions, the full saved pipeline (preprocessing + feature ordering +
graph + model) is run end-to-end on the held-out **known-labelled test set** (8,841 transactions)
and the recomputed metrics are compared against the values stored in `metadata.json`. If they
differ by more than a negligible tolerance, this cell stops rather than proceeding.

In [6]:
print("\n" + "="*70)
print("STEP 5: VERIFY FINAL PIPELINE (known-labelled reproducibility check)")
print("="*70)

with torch.no_grad():
    full_logits = model(x_infer, edge_index)
assert not torch.isnan(full_logits).any().item(), "NaN in model outputs!"
assert not torch.isinf(full_logits).any().item(), "Inf in model outputs!"
print("[PASS] No NaN/Inf in model outputs (full graph forward pass).")

full_probs = F.softmax(full_logits, dim=1)[:, 1].cpu().numpy()

test_probs = full_probs[test_mask_np]
test_true = y_all[test_mask_np]
test_pred = (test_probs >= VALIDATED_THRESHOLD).astype(int)
recheck_precision = precision_score(test_true, test_pred, zero_division=0)
recheck_recall = recall_score(test_true, test_pred, zero_division=0)
recheck_f1 = f1_score(test_true, test_pred, zero_division=0)
recheck_roc_auc = roc_auc_score(test_true, test_probs)
recheck_pr_auc = average_precision_score(test_true, test_probs)

print(f"Verification sample: full held-out test set (n={test_mask_np.sum()} known-labelled transactions)")
print(f"{'Metric':15s} {'Saved (metadata.json)':>22s} {'Recomputed now':>18s} {'|diff|':>10s}")
checks = [
    ("Precision", metadata["test_precision"], recheck_precision),
    ("Recall", metadata["test_recall"], recheck_recall),
    ("F1", metadata["test_f1"], recheck_f1),
    ("ROC-AUC", metadata["test_roc_auc"], recheck_roc_auc),
    ("PR-AUC", metadata["test_pr_auc"], recheck_pr_auc),
]
max_diff = 0.0
for name, saved_v, now_v in checks:
    d = abs(saved_v - now_v)
    max_diff = max(max_diff, d)
    print(f"{name:15s} {saved_v:22.6f} {now_v:18.6f} {d:10.2e}")

if max_diff > 1e-4:
    raise AssertionError(f"STOP: reproducibility mismatch detected (max diff={max_diff:.2e}). Investigate before proceeding.")
print(f"\n[PASS] Final pipeline (saved preprocessing + saved feature ordering + saved graph + saved model) "
      f"reproduces the validated test-set metrics (max diff = {max_diff:.2e}). Safe to proceed to Class-3 inference.")


STEP 5: VERIFY FINAL PIPELINE (known-labelled reproducibility check)


[PASS] No NaN/Inf in model outputs (full graph forward pass).
Verification sample: full held-out test set (n=8841 known-labelled transactions)
Metric           Saved (metadata.json)     Recomputed now     |diff|
Precision                     0.607973           0.607973   0.00e+00
Recall                        0.448529           0.448529   0.00e+00
F1                            0.516220           0.516220   0.00e+00
ROC-AUC                       0.837053           0.837053   0.00e+00
PR-AUC                        0.506344           0.506344   0.00e+00

[PASS] Final pipeline (saved preprocessing + saved feature ordering + saved graph + saved model) reproduces the validated test-set metrics (max diff = 0.00e+00). Safe to proceed to Class-3 inference.


## Step 6 — Class-3 (unknown) inference

Applies the frozen pipeline to all Class-3 transactions. The output is an **illicit-risk score**
(the model's softmax probability for the illicit class) — never described as a confirmed or fraud
probability, since Class 3 is unknown and was never used as ground truth anywhere in this
project.

In [7]:
print("\n" + "="*70)
print("STEP 6: CLASS-3 (UNKNOWN) INFERENCE")
print("="*70)

class3_txids = tx_data.loc[class3_mask_np, "txId"].values
class3_time_steps = tx_data.loc[class3_mask_np, "Time step"].values
class3_scores = full_probs[class3_mask_np]

assert len(class3_txids) == n_class3
print(f"Class-3 transactions scored: {len(class3_txids)} (expected ~157,205)")
print(f"illicit_risk_score range: [{class3_scores.min():.6f}, {class3_scores.max():.6f}]")
print(f"illicit_risk_score mean={class3_scores.mean():.4f}  median={np.median(class3_scores):.4f}")


STEP 6: CLASS-3 (UNKNOWN) INFERENCE
Class-3 transactions scored: 157205 (expected ~157,205)
illicit_risk_score range: [0.000000, 1.000000]
illicit_risk_score mean=0.1962  median=0.0153


## Step 7 — Risk flagging

Uses the already-validated decision threshold (0.90) — not re-tuned here. `HIGH` means the model
assigns a high illicit-risk score based on patterns learned from labelled transactions; it does
**not** mean the transaction is confirmed illicit.

In [8]:
print("\n" + "="*70)
print("STEP 7: RISK FLAGGING (validated threshold = "+str(VALIDATED_THRESHOLD)+", not tuned)")
print("="*70)

risk_level = np.where(class3_scores >= VALIDATED_THRESHOLD, "HIGH", "NOT_HIGH")
flagged = class3_scores >= VALIDATED_THRESHOLD
n_high = int(flagged.sum())
pct_high = n_high / len(class3_scores) * 100
print(f"HIGH means: the model assigns a high illicit-risk score based on patterns learned from labelled "
      f"transactions. It does NOT mean the transaction is confirmed illicit.")
print(f"Transactions flagged HIGH (score >= {VALIDATED_THRESHOLD}): {n_high} ({pct_high:.2f}% of Class-3)")


STEP 7: RISK FLAGGING (validated threshold = 0.9, not tuned)
HIGH means: the model assigns a high illicit-risk score based on patterns learned from labelled transactions. It does NOT mean the transaction is confirmed illicit.
Transactions flagged HIGH (score >= 0.9): 12344 (7.85% of Class-3)


## Step 8 — Save complete Class-3 results

Saved to `outputs/unknown_transaction_risk_scores.csv`, sorted descending by
`illicit_risk_score`, with integrity checks (no duplicate `txId`, no NaN/Inf, scores in [0, 1],
row count equal to the Class-3 count).

In [9]:
print("\n" + "="*70)
print("STEP 8: SAVE COMPLETE CLASS-3 RESULTS")
print("="*70)

results_df = pd.DataFrame({
    "txId": class3_txids,
    "Time step": class3_time_steps,
    "illicit_risk_score": class3_scores,
    "risk_level": risk_level,
    "flagged_at_validation_threshold": flagged,
})
results_df = results_df.sort_values("illicit_risk_score", ascending=False).reset_index(drop=True)

assert results_df["txId"].duplicated().sum() == 0, "Duplicate txId found!"
assert results_df["illicit_risk_score"].isna().sum() == 0, "NaN scores found!"
assert np.isinf(results_df["illicit_risk_score"].values).sum() == 0, "Infinite scores found!"
assert (results_df["illicit_risk_score"] >= 0).all() and (results_df["illicit_risk_score"] <= 1).all(), "Scores out of [0,1]!"
assert len(results_df) == n_class3, "Row count does not equal Class-3 count!"
print("[PASS] No duplicate txId, no NaN, no Inf, all scores in [0,1], row count == Class-3 count.")

risk_scores_path = os.path.join(OUTPUTS_DIR, "unknown_transaction_risk_scores.csv")
results_df.to_csv(risk_scores_path, index=False)
print(f"Saved: {risk_scores_path}  ({len(results_df)} rows)")


STEP 8: SAVE COMPLETE CLASS-3 RESULTS


[PASS] No duplicate txId, no NaN, no Inf, all scores in [0,1], row count == Class-3 count.


Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\unknown_transaction_risk_scores.csv  (157205 rows)


## Step 9 — Forensic case selection (top 20)

The 20 highest-scoring Class-3 transactions, saved to `outputs/top_unknown_forensic_cases.csv`.
These are **candidate cases for forensic investigation**, not confirmed illicit transactions.

In [10]:
print("\n" + "="*70)
print("STEP 9: FORENSIC CASE SELECTION (top 20 highest-scoring Class-3 transactions)")
print("="*70)

top20 = results_df.head(20).copy().reset_index(drop=True)
top20.insert(0, "rank", range(1, len(top20) + 1))
top20_out = top20[["rank", "txId", "Time step", "illicit_risk_score", "risk_level"]]
top_cases_path = os.path.join(OUTPUTS_DIR, "top_unknown_forensic_cases.csv")
top20_out.to_csv(top_cases_path, index=False)
print(f"Saved: {top_cases_path}  ({len(top20_out)} rows)")
print(top20_out.to_string(index=False))
print("\nThese are CANDIDATE CASES for forensic investigation, not confirmed illicit transactions.")


STEP 9: FORENSIC CASE SELECTION (top 20 highest-scoring Class-3 transactions)
Saved: C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\top_unknown_forensic_cases.csv  (20 rows)
 rank      txId  Time step  illicit_risk_score risk_level
    1  99942047         42                 1.0       HIGH
    2  99942248         42                 1.0       HIGH
    3  99942236         42                 1.0       HIGH
    4  99942037         42                 1.0       HIGH
    5  99943929         42                 1.0       HIGH
    6  99942297         42                 1.0       HIGH
    7  99942066         42                 1.0       HIGH
    8 403235564         28                 1.0       HIGH
    9 298977429         30                 1.0       HIGH
   10  99942303         42                 1.0       HIGH
   11 112618841         32                 1.0       HIGH
   12  99943087         42                 1.0       HIGH
   13  99943310         42                 1.0       HIG

## Step 10 — Supporting behavioural indicators

For each top-20 transaction, the raw (pre-scaling, human-readable) values of the final B
behavioural features are retrieved and organized into Financial / Structural / Temporal / Network
groups. These are described as **supporting behavioural indicators**, not causal explanations —
no formal attribution method (e.g. saliency, SHAP) has been applied in this project, so nothing
here should be read as model attribution.

In [11]:
print("\n" + "="*70)
print("STEP 10: SUPPORTING BEHAVIOURAL INDICATORS FOR TOP 20 CASES")
print("="*70)
print("(Raw, pre-scaling, human-readable values retrieved from the final B representation.")
print(" Described as supporting behavioural indicators, not causal explanations.)")

indicator_groups = {
    "FINANCIAL": ["FIN3", "FIN4", "FIN5", "FIN6"],
    "STRUCTURAL": ["STR1", "STR2", "num_input_addresses", "num_output_addresses", "in_txs_degree", "out_txs_degree"],
    "TEMPORAL": ["TMP1", "TMP2", "TMP4", "TMP5", "TMP6", "TMP7", "TMP6_is_first_appearance"],
    "NETWORK": ["NET1", "NET2"],
}
merged_raw_indexed = merged_raw.set_index("txId")

top20_indicators = top20_out.merge(
    merged_raw_indexed[sum(indicator_groups.values(), [])].reset_index(), on="txId", how="left"
)
assert top20_indicators.shape[0] == 20
print(f"Retrieved {len(sum(indicator_groups.values(), []))} raw indicator columns for {len(top20_indicators)} top cases.")
print("\nSample (rank 1):")
r1 = top20_indicators.iloc[0]
for grp, cols in indicator_groups.items():
    print(f"  {grp}: " + ", ".join(f"{c}={r1[c]:.4g}" if pd.notna(r1[c]) else f"{c}=N/A" for c in cols))


STEP 10: SUPPORTING BEHAVIOURAL INDICATORS FOR TOP 20 CASES
(Raw, pre-scaling, human-readable values retrieved from the final B representation.
 Described as supporting behavioural indicators, not causal explanations.)


Retrieved 19 raw indicator columns for 20 top cases.

Sample (rank 1):
  FINANCIAL: FIN3=1, FIN4=1, FIN5=0.0315, FIN6=7.244e-06
  STRUCTURAL: STR1=1, STR2=0, num_input_addresses=1, num_output_addresses=1, in_txs_degree=1, out_txs_degree=0
  TEMPORAL: TMP1=0, TMP2=0, TMP4=0, TMP5=13, TMP6=N/A, TMP7=1, TMP6_is_first_appearance=1
  NETWORK: NET1=0, NET2=13


## Step 11 — Forensic report table

A human-readable report for the top candidate cases, grouped by indicator category, closing with
the standard interpretation disclaimer. Shown here for the top 5 (all 20 are available in
`top20_indicators` for further review).

In [12]:
print("\n" + "="*70)
print("STEP 11: FORENSIC REPORT TABLE (top 5 shown in full report format)")
print("="*70)

def fmt(v):
    return f"{v:.4g}" if pd.notna(v) else "N/A (no qualifying prior activity)"

report_rows = []
for i, row in top20_indicators.head(5).iterrows():
    lines = []
    lines.append(f"Transaction ID: {row['txId']}")
    lines.append(f"Time step: {row['Time step']}")
    lines.append(f"Risk Score: {row['illicit_risk_score']:.4f}")
    lines.append(f"Risk Level: {row['risk_level']}")
    lines.append("")
    lines.append("Financial:")
    lines.append(f"  - largest input share (FIN3) = {fmt(row['FIN3'])}")
    lines.append(f"  - largest output share (FIN4) = {fmt(row['FIN4'])}")
    lines.append(f"  - fee-to-value ratio (FIN5) = {fmt(row['FIN5'])}")
    lines.append(f"  - fee rate (FIN6) = {fmt(row['FIN6'])}")
    lines.append("Structural:")
    lines.append(f"  - input/output address ratio (STR1) = {fmt(row['STR1'])}")
    lines.append(f"  - same-tx address reuse (STR2) = {fmt(row['STR2'])}")
    lines.append(f"  - num_input_addresses = {fmt(row['num_input_addresses'])}")
    lines.append(f"  - num_output_addresses = {fmt(row['num_output_addresses'])}")
    lines.append(f"  - in_txs_degree = {fmt(row['in_txs_degree'])}")
    lines.append(f"  - out_txs_degree = {fmt(row['out_txs_degree'])}")
    lines.append("Temporal:")
    lines.append(f"  - input-address age (TMP1) = {fmt(row['TMP1'])}")
    lines.append(f"  - output-address age (TMP2) = {fmt(row['TMP2'])}")
    lines.append(f"  - prior sender tx count (TMP4) = {fmt(row['TMP4'])}")
    lines.append(f"  - prior receiver tx count (TMP5) = {fmt(row['TMP5'])}")
    lines.append(f"  - time since last activity (TMP6) = {fmt(row['TMP6'])}")
    lines.append(f"  - distinct active time steps (TMP7) = {fmt(row['TMP7'])}")
    lines.append(f"  - first appearance flag = {fmt(row['TMP6_is_first_appearance'])}")
    lines.append("Network:")
    lines.append(f"  - historical linked tx count, input side (NET1) = {fmt(row['NET1'])}")
    lines.append(f"  - historical linked tx count, output side (NET2) = {fmt(row['NET2'])}")
    lines.append("")
    lines.append("Interpretation: This transaction exhibits behavioural characteristics associated with")
    lines.append("patterns learned by the model. These indicators support forensic review but do not")
    lines.append("establish illicit activity.")
    lines.append("-"*70)
    report_rows.append("\n".join(lines))

print("\n\n".join(report_rows))


STEP 11: FORENSIC REPORT TABLE (top 5 shown in full report format)
Transaction ID: 99942047
Time step: 42
Risk Score: 1.0000
Risk Level: HIGH

Financial:
  - largest input share (FIN3) = 1
  - largest output share (FIN4) = 1
  - fee-to-value ratio (FIN5) = 0.0315
  - fee rate (FIN6) = 7.244e-06
Structural:
  - input/output address ratio (STR1) = 1
  - same-tx address reuse (STR2) = 0
  - num_input_addresses = 1
  - num_output_addresses = 1
  - in_txs_degree = 1
  - out_txs_degree = 0
Temporal:
  - input-address age (TMP1) = 0
  - output-address age (TMP2) = 0
  - prior sender tx count (TMP4) = 0
  - prior receiver tx count (TMP5) = 13
  - time since last activity (TMP6) = N/A (no qualifying prior activity)
  - distinct active time steps (TMP7) = 1
  - first appearance flag = 1
Network:
  - historical linked tx count, input side (NET1) = 0
  - historical linked tx count, output side (NET2) = 13

Interpretation: This transaction exhibits behavioural characteristics associated with
patte

## Step 12 — Final notebook output: completion summary

In [13]:
print("\n" + "="*70)
print("COMPLETION SUMMARY")
print("="*70)
print(f"Checkpoint loaded:            {os.path.join(ARTIFACT_DIR, 'model_checkpoint.pt')}")
print(f"Preprocessing loaded:         quantile_transformer.joblib, median_imputation.joblib, "
      f"feature_columns.json, scaling_columns.json, binary_columns.json, metadata.json")
print(f"Feature count:                {x_infer.shape[1]} (expected 186)")
print(f"Graph nodes:                  {n_nodes} (expected 203,769)")
print(f"Graph edges:                  {edge_index.shape[1]} (expected 234,355)")
print(f"Class-3 (unknown) count:      {n_class3}")
print(f"High-risk count:              {n_high}")
print(f"High-risk percentage:         {pct_high:.2f}%")
print(f"Output files created:")
print(f"  - {risk_scores_path}")
print(f"  - {top_cases_path}")
print(f"\nTotal runtime: {time.time()-t0:.1f}s")


COMPLETION SUMMARY
Checkpoint loaded:            C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\gnn_B_quantile_seed42\model_checkpoint.pt
Preprocessing loaded:         quantile_transformer.joblib, median_imputation.joblib, feature_columns.json, scaling_columns.json, binary_columns.json, metadata.json
Feature count:                186 (expected 186)
Graph nodes:                  203769 (expected 203,769)
Graph edges:                  234355 (expected 234,355)
Class-3 (unknown) count:      157205
High-risk count:              12344
High-risk percentage:         7.85%
Output files created:
  - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\unknown_transaction_risk_scores.csv
  - C:\Users\koush\OneDrive\Documents\Financial_Forensics\outputs\top_unknown_forensic_cases.csv

Total runtime: 23.7s
